# Panel de actualización de Inmobil-IA-ria

Este notebook ejecuta el flujo completo mostrando el avance en pantalla:

1. Recorre la búsqueda de venta en Zaragoza ordenada por anuncios recientes.
2. Se detiene tras 5 IDs conocidos consecutivos y scrapea los datos de los IDs nuevos.
3. Procesa los inmuebles y calcula las predicciones.
4. Abre el CSV de oportunidades y espera a que lo guardes.
5. Obtiene título y portada de los seleccionados, genera sus informes y los envía a los clientes correspondientes.

> Ejecuta **Run All**. El notebook se detendrá únicamente mientras editas el CSV.

In [ ]:
from pathlib import Path
import sys

from IPython.display import FileLink, display
import pandas as pd
from rich.console import Console


def encontrar_raiz(inicio):
    inicio = Path(inicio).resolve()
    for candidata in (inicio, *inicio.parents):
        if (candidata / 'pyproject.toml').exists():
            return candidata
    raise RuntimeError('No se encontró la raíz del proyecto')


RAIZ = encontrar_raiz(Path.cwd())
sys.path.insert(0, str(RAIZ / 'src'))
for nombre_modulo in [n for n in sys.modules if n == 'inmobil_iaria' or n.startswith('inmobil_iaria.')]:
    del sys.modules[nombre_modulo]

from inmobil_iaria.config import AppConfig
from inmobil_iaria.sources.idealista_browser import browser_source_from_config
from inmobil_iaria.workflow import collect_new_properties
from inmobil_iaria.matching import read_review, run_matching, selected_properties
from inmobil_iaria.notifications import notification_preview, send_notifications
from inmobil_iaria.prediction import run_prediction
from inmobil_iaria.preprocessing import run_preprocessing
from inmobil_iaria.reporting import generate_reports, prepare_report_assets
from inmobil_iaria.review import file_version, open_review, wait_until_saved
from inmobil_iaria.storage import RunState

CONFIG = AppConfig.from_env(RAIZ)
console = Console()
console.print(f'[green]Proyecto cargado:[/green] {RAIZ}')

## Configuración de la ejecución

- `auto`: continúa una revisión pendiente; si no existe, inicia un rastreo completo.
- `completo`: importa anuncios nuevos y ejecuta todo el proceso.
- `reutilizar_ids`: repite el proceso con los IDs de la última ejecución, sin rastrear.
- `continuar_revision`: abre el CSV que ya existe y genera los informes al guardarlo.

La fuente es el scraper de la búsqueda de venta en Zaragoza. El límite de IDs conocidos consecutivos se configura en `.env`; tras tu aprobación se obtienen la portada y el título antes de crear y enviar los informes.

In [ ]:
MODO = 'auto'  # auto | completo | reutilizar_ids | continuar_revision
COMPILAR_PDF = True
ENVIAR_CORREOS = False  # habilitar solo tras una revision explicita

## Ejecutar el flujo

Esta celda permanece activa mientras el CSV está abierto. Marca `Seleccionado` como `Sí` en las oportunidades deseadas y guarda el archivo para continuar.

In [ ]:
def revisar_y_generar_informes():
    ruta_revision = CONFIG.paths.review
    if not ruta_revision.exists():
        console.print('[red]No existe todavía un CSV de oportunidades para revisar.[/red]')
        return []

    oportunidades = read_review(ruta_revision)
    if oportunidades.empty:
        console.print('[yellow]No hay oportunidades que revisar.[/yellow]')
        return []

    columnas = [
        columna for columna in (
            'id', 'Localizacion', 'Precio', 'Diferencia_Ponderada',
            'Clientes_Interesados', 'Seleccionado', 'Enlace'
        ) if columna in oportunidades.columns
    ]
    console.rule(f'[bold cyan]{len(oportunidades)} oportunidades para revisar')
    display(oportunidades[columnas])

    version_inicial = file_version(ruta_revision)
    open_review(ruta_revision)
    console.print(f'[bold yellow]CSV abierto:[/bold yellow] {ruta_revision}')
    console.print('Marca Seleccionado como Sí, guarda el archivo y vuelve al notebook.')
    console.print('[yellow]⏳ Esperando a que guardes el CSV...[/yellow]')
    wait_until_saved(ruta_revision, version_inicial)

    revisadas = read_review(ruta_revision)
    seleccionadas = selected_properties(revisadas)
    console.print(f'[green]✅ Guardado detectado: {len(seleccionadas)} seleccionadas[/green]')
    if seleccionadas.empty:
        estado = RunState.load(CONFIG.paths.run_state)
        estado.status = 'review_completed_no_selection'
        estado.save(CONFIG.paths.run_state)
        console.print('[yellow]No has seleccionado ningún inmueble. El flujo termina aquí.[/yellow]')
        return []

    display(seleccionadas[columnas])
    console.rule('[bold cyan]Título y portada de los seleccionados')
    prepare_report_assets(CONFIG, progress=console.print)
    console.rule('[bold cyan]Generación de informes')
    informes = generate_reports(CONFIG.paths, compile_pdf=COMPILAR_PDF)

    estado = RunState.load(CONFIG.paths.run_state)
    estado.generated_reports = [str(informe) for informe in informes]
    estado.status = 'reports_generated'
    estado.save(CONFIG.paths.run_state)

    console.print(f'[bold green]✅ {len(informes)} informes generados[/bold green]')
    for informe in informes:
        display(FileLink(str(informe)))

    if ENVIAR_CORREOS:
        enviados = send_notifications(CONFIG)
        console.print(f'[bold green]✉️ Correos enviados: {enviados}[/bold green]')
    else:
        vista_previa = notification_preview(CONFIG)
        console.print(
            f'[cyan]Envío desactivado. {len(vista_previa)} clientes recibirían informes.[/cyan]'
        )

    return informes


def ejecutar_actualizacion():
    if MODO not in {'auto', 'completo', 'reutilizar_ids', 'continuar_revision'}:
        raise ValueError(f'MODO no válido: {MODO}')

    estado = RunState.load(CONFIG.paths.run_state)
    modo_efectivo = MODO
    if MODO == 'auto':
        hay_revision = CONFIG.paths.review.exists() and estado.status == 'awaiting_review'
        modo_efectivo = 'continuar_revision' if hay_revision else 'completo'
        console.print(f'[cyan]Modo automático: {modo_efectivo}[/cyan]')

    if modo_efectivo == 'continuar_revision':
        console.rule('[bold cyan]Continuando la revisión existente')
        return revisar_y_generar_informes()

    if modo_efectivo == 'completo':
        console.rule('[bold cyan]1. Entrada de anuncios nuevos')
        fuente = browser_source_from_config(CONFIG, progress=console.print)
        nuevos_ids = collect_new_properties(CONFIG.paths, fuente)
        estado.new_ids = nuevos_ids
        estado.status = 'scraped'
        estado.save(CONFIG.paths.run_state)
    else:
        nuevos_ids = estado.new_ids
        console.print(f'[cyan]Reutilizando {len(nuevos_ids)} IDs de la última ejecución.[/cyan]')

    if not nuevos_ids:
        console.print('[yellow]No se han encontrado inmuebles nuevos.[/yellow]')
        return []

    console.rule(f'[bold cyan]{len(nuevos_ids)} IDs añadidos')
    display(pd.DataFrame({'id': nuevos_ids}))

    console.rule('[bold cyan]2. Preparación de datos')
    procesadas = run_preprocessing(CONFIG.paths)
    console.print(f'[green]✅ Histórico consolidado: {len(procesadas)} inmuebles[/green]')

    console.rule('[bold cyan]3. Predicciones')
    predicciones = run_prediction(CONFIG.paths)
    console.print(f'[green]✅ Predicciones calculadas: {len(predicciones)} inmuebles[/green]')

    console.rule('[bold cyan]4. Cruce con clientes')
    oportunidades = run_matching(CONFIG.paths, nuevos_ids)
    estado.status = 'awaiting_review'
    estado.save(CONFIG.paths.run_state)
    console.print(f'[bold green]✅ Oportunidades encontradas: {len(oportunidades)}[/bold green]')

    if oportunidades.empty:
        console.print('[yellow]No hay oportunidades que cumplan los filtros.[/yellow]')
        return []

    return revisar_y_generar_informes()


informes_generados = ejecutar_actualizacion()